%pip install -U -q numpy pandas wfdb matplotlib notebook scikit-learn PyWavelets plotly

In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import wfdb
import matplotlib.pyplot as plt
import numpy as np
import os
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression

from funkcje import *
from processor import Signal, ICP_ABP_Processor

# Sekcja wczytywania

In [ ]:
%reload_ext autoreload
data_dir = 'charis-database-1.0.0'  # Lokalny folder
record_name = 'charis3'     # Nazwa rekordu (.hea i .dat)

os.makedirs(data_dir, exist_ok=True)

# TERAZ TWÓJ KOD BEZ ZMIAN
local_path = os.path.join(data_dir, record_name)

# Odczyt (automatycznie znajdzie .hea i .dat)
record = wfdb.rdrecord(local_path)

# Dane sygnałów
signals = record.p_signal  # NxM tablica (próbki x kanały)
fs = record.fs  # Częstotliwość próbkowania
time = np.arange(len(signals)) / fs # sekundy

print(f"Rekord: {record_name}")
print(f"Długość: {len(signals)} próbek ({len(signals)/fs/3600:.1f} h)")
print(f"fs: {fs} Hz, Kanały: {record.n_sig}")
print(f"Nazwy kanałów: {record.sig_name}")  # ['ABP', 'ECG', 'ICP']


channel_ABP = signals[:,0]
channel_ICP = signals[:,2]

# Inicjalizacja
processor = ICP_ABP_Processor(channel_ICP, channel_ABP, sampling_freq=fs)
processor.process_all()


In [ ]:
# Wizualizacja pierwszych 60s
plt.figure(figsize=(15, 6))
plt.plot(time[int(6000*fs):int(12000*fs)], processor.icp.data[int(6000*fs):int(12000*fs)], linewidth=0.8)
plt.xlabel('Czas [s]')
plt.ylabel('Amplituda [mmHg]')
plt.title(f'{record_name} kanał ICP')
plt.grid(True, alpha=0.3)
plt.show()

# Wizualizacja pierwszych 60s
plt.figure(figsize=(15, 6))
plt.plot(time[int(6000*fs):int(12000*fs)], processor.abp.data[int(6000*fs):int(12000*fs)], linewidth=0.8)
plt.xlabel('Czas [s]')
plt.ylabel('Amplituda [mmHg]')
plt.title(f'{record_name} kanał ABP')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
x_seconds_avg = 5
window_size = 60

# 1. Pobranie danych pogrupowanych w okna
okna_icp, okna_abp = processor.get_windowed_data(x_seconds_avg=x_seconds_avg, window_size=window_size)

# 2. Obliczenie PRx z poprawnym mapowaniem czasu rzeczywistego
wyniki_prx = calculate_PRx(okna_icp, okna_abp, x_seconds_avg=x_seconds_avg)

eventPRx, normalPRx = defineTimeZones(PRx=wyniki_prx, start= , end= , pair_time_sectors=)


# 3. Przygotowanie danych do wykresu
czas_w_godzinach = wyniki_prx[:, 0]
wartosci_prx = wyniki_prx[:, 1]

# Przeliczamy godziny na minuty tylko dla osi wykresu
czas_w_minutach = czas_w_godzinach * 60 

# Tworzymy precyzyjne opisy dymków "X min Y s"
minuty_pelne = np.floor(czas_w_minutach)
sekundy_reszta = np.round((czas_w_minutach - minuty_pelne) * 60)
hover_text = [f"{int(m)} min {int(s):02d} s" for m, s in zip(minuty_pelne, sekundy_reszta)]

# --- 4. RYSUJ INTERAKTYWNY WYKRES (PLOTLY) ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=czas_w_minutach,  # Prawidłowa oś czasowa w minutach (dla 95h sięgnie ~5700)
    y=wartosci_prx,
    mode='lines',
    name='PRx',
    line=dict(color='blue', width=2),
    text=hover_text,
    hovertemplate='<b>Czas:</b> %{text}<br><b>PRx:</b> %{y:.3f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='Monitorowanie indeksu PRx w czasie (pełny sygnał)',
        x=0.5,
        font=dict(size=16)
    ),
    xaxis=dict(
        title='Czas (minuty)',
        gridcolor='rgba(200, 200, 200, 0.2)',
        showspikes=True,
        spikemode='across',
        spikethickness=1,
        spikedash='dash'
    ),
    yaxis=dict(
        title='Wartość PRx',
        range=[-1.1, 1.1],
        gridcolor='rgba(200, 200, 200, 0.2)',
        zeroline=True,
        zerolinecolor='rgba(0, 0, 0, 0.3)',
        zerolinewidth=1
    ),
    template='plotly_white',
    hovermode='x unified',
    dragmode='zoom',
    width=1600,
    height=400
)

fig.show()

In [ ]:
x_seconds_avg = 10
window_size = 30

# 1. Pobranie danych pogrupowanych w okna
okna_icp, okna_abp = processor.get_windowed_data(x_seconds_avg=x_seconds_avg, window_size=window_size)

# 2. Obliczenie PRx z poprawnym mapowaniem czasu rzeczywistego
wyniki_prx = calculate_PRx(okna_icp, okna_abp, x_seconds_avg=x_seconds_avg)

# 3. Przygotowanie danych do wykresu
czas_w_godzinach = wyniki_prx[:, 0]
wartosci_prx = wyniki_prx[:, 1]

# Przeliczamy godziny na minuty tylko dla osi wykresu
czas_w_minutach = czas_w_godzinach * 60 

# Tworzymy precyzyjne opisy dymków "X min Y s"
minuty_pelne = np.floor(czas_w_minutach)
sekundy_reszta = np.round((czas_w_minutach - minuty_pelne) * 60)
hover_text = [f"{int(m)} min {int(s):02d} s" for m, s in zip(minuty_pelne, sekundy_reszta)]

# --- 4. RYSUJ INTERAKTYWNY WYKRES (PLOTLY) ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=czas_w_minutach,  # Prawidłowa oś czasowa w minutach (dla 95h sięgnie ~5700)
    y=wartosci_prx,
    mode='lines',
    name='PRx',
    line=dict(color='blue', width=2),
    text=hover_text,
    hovertemplate='<b>Czas:</b> %{text}<br><b>PRx:</b> %{y:.3f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='Monitorowanie indeksu PRx w czasie (pełny sygnał)',
        x=0.5,
        font=dict(size=16)
    ),
    xaxis=dict(
        title='Czas (minuty)',
        gridcolor='rgba(200, 200, 200, 0.2)',
        showspikes=True,
        spikemode='across',
        spikethickness=1,
        spikedash='dash'
    ),
    yaxis=dict(
        title='Wartość PRx',
        range=[-1.1, 1.1],
        gridcolor='rgba(200, 200, 200, 0.2)',
        zeroline=True,
        zerolinecolor='rgba(0, 0, 0, 0.3)',
        zerolinewidth=1
    ),
    template='plotly_white',
    hovermode='x unified',
    dragmode='zoom',
    width=1600,
    height=400
)

fig.show()

In [ ]:
x_seconds_avg = 15
window_size = 20

# 1. Pobranie danych pogrupowanych w okna
okna_icp, okna_abp = processor.get_windowed_data(x_seconds_avg=x_seconds_avg, window_size=window_size)

# 2. Obliczenie PRx z poprawnym mapowaniem czasu rzeczywistego
wyniki_prx = calculate_PRx(okna_icp, okna_abp, x_seconds_avg=x_seconds_avg)

# 3. Przygotowanie danych do wykresu
czas_w_godzinach = wyniki_prx[:, 0]
wartosci_prx = wyniki_prx[:, 1]

# Przeliczamy godziny na minuty tylko dla osi wykresu
czas_w_minutach = czas_w_godzinach * 60 

# Tworzymy precyzyjne opisy dymków "X min Y s"
minuty_pelne = np.floor(czas_w_minutach)
sekundy_reszta = np.round((czas_w_minutach - minuty_pelne) * 60)
hover_text = [f"{int(m)} min {int(s):02d} s" for m, s in zip(minuty_pelne, sekundy_reszta)]

# --- 4. RYSUJ INTERAKTYWNY WYKRES (PLOTLY) ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=czas_w_minutach,  # Prawidłowa oś czasowa w minutach (dla 95h sięgnie ~5700)
    y=wartosci_prx,
    mode='lines',
    name='PRx',
    line=dict(color='blue', width=2),
    text=hover_text,
    hovertemplate='<b>Czas:</b> %{text}<br><b>PRx:</b> %{y:.3f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='Monitorowanie indeksu PRx w czasie (pełny sygnał)',
        x=0.5,
        font=dict(size=16)
    ),
    xaxis=dict(
        title='Czas (minuty)',
        gridcolor='rgba(200, 200, 200, 0.2)',
        showspikes=True,
        spikemode='across',
        spikethickness=1,
        spikedash='dash'
    ),
    yaxis=dict(
        title='Wartość PRx',
        range=[-1.1, 1.1],
        gridcolor='rgba(200, 200, 200, 0.2)',
        zeroline=True,
        zerolinecolor='rgba(0, 0, 0, 0.3)',
        zerolinewidth=1
    ),
    template='plotly_white',
    hovermode='x unified',
    dragmode='zoom',
    width=1600,
    height=400
)

fig.show()